# Notebook 02: Cyclone Fani case study (Workstream 1B) and fragility calibration
**What this notebook does**
1. Rebuilds Fani's wind field from the IMD best track (3-minute winds), using a Holland (1980) profile with translation asymmetry
2. Converts it to the **3-second gust at 10 m** at every tower using the WMO/TD-1555 gust factor for each tower's exposure class (Notebook 01)
3. Checks the modelled winds against station observations (Meteostat / NOAA ISD)
4. Compares the modelled gust at the 128 damaged towers with the design wind (IS 875 50 m/s gust; OPTCL's IS 802:1995 design wind of 39.3 m/s 10-min)
5. **Fits lognormal fragility curves** (any damage, and collapse) by maximum likelihood, on the same wind measure the hazard notebooks use, so hazard and fragility are consistent
6. Tests how sensitive the fit is to the radius of maximum wind, Holland B and the exposure assumption

**Needs:** Notebooks 00 (required) and 01 (optional; without it a distance-to-coast rule sets exposure).

In [ ]:
# ==== Shared functions (cdri_lib): identical copy in every notebook from 01 onwards ====
# Change a function here, rebuild the notebooks, and every notebook uses the same version.
import os, sys, re, json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import minimize
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 160)

KT2MS, NMI2KM, R_EARTH_KM = 0.514444, 1.852, 6371.0
SURF_TO_GRADIENT = 0.85          # surface-to-gradient wind ratio used only to estimate the Holland B parameter
CRS_GEO, CRS_M = 'EPSG:4326', 'EPSG:32645'

# WMO/TD-1555 (Harper, Kepert & Ginger 2010) Table 1.1: 3-s gust factor G(3, To) at +10 m in tropical cyclones
GUST3 = {'in-land':  {60: 1.49, 180: 1.58, 600: 1.66},
         'off-land': {60: 1.36, 180: 1.44, 600: 1.52},
         'off-sea':  {60: 1.23, 180: 1.31, 600: 1.38},
         'at-sea':   {60: 1.11, 180: 1.17, 600: 1.23}}


def setup_project():
    """Mount Google Drive (in Colab) and return the project root and folder dictionary."""
    if 'google.colab' in sys.modules:
        from google.colab import drive
        try:
            drive.mount('/content/drive')
            root = Path('/content/drive/MyDrive/CDRI_Odisha')
        except Exception as e:
            print('Drive mount failed:', e, '\nFix: Runtime → Disconnect and delete runtime, rerun, use the same Google '
                  'account and tick all permission boxes. Using temporary storage for now.')
            root = Path('/content/CDRI_Odisha')
    else:
        root = Path(os.environ.get('CDRI_ROOT', Path.cwd() / 'CDRI_Odisha'))
    dirs = {k: root / v for k, v in {'raw': 'data/raw', 'external': 'data/external', 'interim': 'data/interim',
                                     'processed': 'data/processed', 'figures': 'outputs/figures',
                                     'tables': 'outputs/tables'}.items()}
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    print('Project root:', root)
    return root, dirs


def union(gs):
    """Dissolve a GeoSeries/GeoDataFrame to one geometry (old and new geopandas)."""
    return gs.union_all() if hasattr(gs, 'union_all') else gs.unary_union


def need(path, notebook):
    """Stop with a clear message if an input from an earlier notebook is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(f'{path} not found. Run Notebook {notebook} first.')
    return Path(path)


def gust_factor(exposure, avg_s):
    """WMO 3-s gust factor for each exposure class, given the averaging period (s) of the mean wind."""
    return np.array([GUST3[e][int(avg_s)] for e in np.asarray(exposure, dtype=object)], dtype=float)


def haversine_km(lat1, lon1, lat2, lon2):
    la1, lo1, la2, lo2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * R_EARTH_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def _interp_valid(tn, t, y):
    y = np.asarray(y, float)
    ok = np.isfinite(y)
    if ok.sum() == 0:
        return np.full(len(tn), np.nan)
    if ok.sum() == 1:
        return np.full(len(tn), y[ok][0])
    return np.interp(tn, t[ok], y[ok])


def interp_track(t_h, lat, lon, vmax, rmw, pc=None, dt_h=1.0):
    """Interpolate a track to dt_h-hour steps and add the translation velocity (u east, v north; m/s)."""
    t_h = np.asarray(t_h, float)
    if len(t_h) < 2:
        return None
    tn = np.arange(t_h[0], t_h[-1] + 1e-9, dt_h)
    tr = {'t_h': tn, 'lat': _interp_valid(tn, t_h, lat), 'lon': _interp_valid(tn, t_h, lon),
          'vmax': _interp_valid(tn, t_h, vmax), 'rmw': _interp_valid(tn, t_h, rmw),
          'pc': _interp_valid(tn, t_h, pc) if pc is not None else np.full(len(tn), np.nan)}
    if len(tn) < 2:
        tr['u'] = tr['v'] = np.zeros(len(tn))
        return tr
    dy = np.gradient(tr['lat']) * 111.195
    dx = np.gradient(tr['lon']) * 111.195 * np.cos(np.radians(tr['lat']))
    dt = np.gradient(tn) * 3600.0
    tr['u'], tr['v'] = dx * 1000 / dt, dy * 1000 / dt
    return tr


def holland_max_wind(tr, plat, plon, p_env=1008.0, b_fixed=None, b_default=1.6, rho=1.15,
                     max_dist_km=600.0, chunk=4000):
    """Storm-maximum surface wind at points (same averaging basis as tr['vmax']).

    Holland (1980) radial profile scaled to the symmetric maximum wind, plus a translation asymmetry
    (Jelesnianski 1966: Vt·Rm·r/(Rm²+r²), half the translation speed at the RMW, strongest right of track).
    B comes from the central-pressure deficit (Holland 1980) when pressure is known, else b_default; clipped to 1.0–2.5.
    No land-roughness reduction is applied (as in STORM / Bloemendaal 2020); exposure is handled by the gust factor.
    """
    plat, plon = np.asarray(plat, float), np.asarray(plon, float)
    out = np.zeros(len(plat))
    lat, lon = tr['lat'][:, None], tr['lon'][:, None]
    u, v = tr['u'][:, None], tr['v'][:, None]
    vm, rm = tr['vmax'][:, None], np.maximum(tr['rmw'][:, None], 5.0)
    vt = np.hypot(u, v)
    vsym = np.maximum(vm - 0.5 * vt, 0.5 * vm)
    if b_fixed is not None:
        B = np.full_like(vm, float(b_fixed))
    else:
        dp = (p_env - tr['pc'][:, None]) * 100.0
        vg = vsym / SURF_TO_GRADIENT
        B = np.where(np.isfinite(dp) & (dp > 200.0), rho * np.e * vg ** 2 / np.maximum(dp, 200.0), b_default)
        B = np.clip(B, 1.0, 2.5)
    coslat = np.cos(np.radians(lat))
    for s in range(0, len(plat), chunk):
        pl, po = plat[None, s:s + chunk], plon[None, s:s + chunk]
        dy = (pl - lat) * 111.195
        dx = (po - lon) * 111.195 * coslat
        r = np.maximum(np.hypot(dx, dy), 0.5)
        x = (rm / r) ** B
        vs = vsym * np.sqrt(x * np.exp(1.0 - x))
        proj = (-dy * u + dx * v) / r                     # translation component along the cyclonic tangential direction
        w = vs + proj * (rm * r / (rm ** 2 + r ** 2))
        w = np.where(r <= max_dist_km, w, 0.0)
        out[s:s + chunk] = np.maximum(w.max(axis=0), 0.0)
    return out


def frag_p(im, theta, beta):
    """Lognormal fragility: P(damage | IM)."""
    im = np.maximum(np.asarray(im, float), 1e-3)
    return norm.cdf((np.log(im) - np.log(theta)) / beta)


def fit_fragility(im, y, theta0=None, beta0=0.2):
    """Maximum-likelihood lognormal fragility from binary outcomes (Baker 2015). Returns theta, beta, -loglik."""
    im, y = np.asarray(im, float), np.asarray(y, bool)
    ok = np.isfinite(im) & (im > 0)
    im, y = im[ok], y[ok]
    if theta0 is None:
        theta0 = np.median(im[y]) if y.any() else np.percentile(im, 99)
    li = np.log(im)

    def nll(p):
        pr = np.clip(norm.cdf((li - p[0]) / np.exp(p[1])), 1e-12, 1 - 1e-12)
        return -(y * np.log(pr) + (~y) * np.log(1 - pr)).sum()
    res = minimize(nll, [np.log(theta0), np.log(beta0)], method='Nelder-Mead',
                   options={'xatol': 1e-7, 'fatol': 1e-7, 'maxiter': 5000})
    return float(np.exp(res.x[0])), float(np.exp(res.x[1])), float(res.fun)


def bootstrap_fragility(im, y, n=200, seed=1):
    rng = np.random.default_rng(seed)
    im, y = np.asarray(im, float), np.asarray(y, bool)
    th, be = [], []
    for _ in range(n):
        i = rng.integers(0, len(im), len(im))
        if y[i].sum() < 3:
            continue
        t, b, _ = fit_fragility(im[i], y[i])
        th.append(t); be.append(b)
    return np.array(th), np.array(be)


# ---------- project data loaders ----------
def load_towers(dirs):
    """Towers from Notebook 00, with exposure from Notebook 01 if available (else a distance-to-coast rule)."""
    t = pd.read_parquet(need(dirs['processed'] / 'towers.parquet', '00'))
    ex = dirs['processed'] / 'towers_exposure.parquet'
    if ex.exists():
        e = pd.read_parquet(ex)
        t = t.merge(e[['tower_id', 'z0', 'terrain_cat', 'exposure', 'exposure_src']], on='tower_id', how='left')
    else:
        t['z0'], t['terrain_cat'], t['exposure'], t['exposure_src'] = np.nan, np.nan, None, None
    miss = t.exposure.isna()
    t.loc[miss, 'exposure'] = np.where(t.loc[miss, 'dist_coast_km'] <= 2.0, 'off-sea', 'in-land')
    t.loc[miss, 'exposure_src'] = 'rule: ≤2 km from coast = off-sea, else in-land'
    t['terrain_cat'] = t.terrain_cat.fillna(2).astype(int)
    return t


def load_network(dirs):
    """Lines and substations (with population served) from Notebooks 00/01."""
    import geopandas as gpd
    gpkg = need(dirs['processed'] / 'odisha_network.gpkg', '00')
    lines = gpd.read_file(gpkg, layer='lines')
    subs = gpd.read_file(gpkg, layer='substations')
    popf = dirs['processed'] / 'substations_population.csv'
    if popf.exists():
        p = pd.read_csv(popf)
        subs = subs.merge(p[['sub_id', 'pop_worldpop']], on='sub_id', how='left')
        subs['pop'] = subs.pop_worldpop.fillna(subs.pop_weight)
        subs['pop_src'] = np.where(subs.pop_worldpop.notna(), 'WorldPop 2020 service area', 'Raj et al. weight')
    else:
        subs['pop'], subs['pop_src'] = subs.pop_weight, 'Raj et al. weight'
    subs['pop'] = subs['pop'].fillna(0.0)
    return lines, subs


def load_fragility(dirs, which='any_damage'):
    f = dirs['processed'] / 'fragility_params.json'
    if f.exists():
        d = json.loads(f.read_text())
        if which in d:
            return d[which]['theta'], d[which]['beta'], d[which].get('source', 'Notebook 02')
    print('  ! fragility_params.json not found: using Raj et al. collapse fragility (292.5 km/h, β = 0.104)')
    return 292.5 / 3.6, 0.104, 'Raj et al. (collapse)'


# ---------- risk engine (Notebooks 05 and 06) ----------
def line_matrix(line_pos, n_lines):
    """Sparse tower × line indicator used to combine tower failure probabilities into line failure probabilities."""
    n = len(line_pos)
    return csr_matrix((np.ones(n), (np.arange(n), line_pos)), shape=(n, n_lines))


def storm_probabilities(H, cell, gfac, theta_t, beta, M, scale=1.0, batch=512):
    """For every storm: expected damaged towers, line failure probabilities and per-tower annual sums.

    H: storms × cells storm-max 10-min wind; cell: grid cell of each tower; gfac: wind→IM factor per tower;
    theta_t: fragility median per tower (array); M: tower × line indicator."""
    n_s, n_l = H.shape[0], M.shape[1]
    E = np.zeros(n_s); P_line = np.zeros((n_s, n_l), np.float32); tower_sum = np.zeros(len(cell))
    lt = np.log(theta_t)
    for s in range(0, n_s, batch):
        im = H[s:s + batch][:, cell].astype(np.float32) * (gfac * scale)[None, :]
        p = norm.cdf((np.log(np.maximum(im, 1e-3)) - lt[None, :]) / beta)
        E[s:s + batch] = p.sum(1)
        tower_sum += p.sum(0)
        logq = (M.T @ np.log1p(-np.minimum(p, 1 - 1e-9)).T).T
        P_line[s:s + batch] = 1 - np.exp(logq)
    return E, P_line, tower_sum


def outage_mc(P_line, frm, to, n_sub, sub_pop, n_mc=30, min_p=1e-3, seed=7):
    """Expected population disconnected per storm (Raj et al.: population outside the largest connected component)."""
    rng = np.random.default_rng(seed)

    def lost(alive):
        g = csr_matrix((np.ones(alive.sum()), (frm[alive], to[alive])), shape=(n_sub, n_sub))
        nc, lab = connected_components(g, directed=False)
        big = np.bincount(lab).argmax()
        return sub_pop[lab != big].sum()
    base = lost(np.ones(len(frm), bool))
    out = np.zeros(P_line.shape[0])
    for s in np.where(P_line.max(1) > min_p)[0]:
        fails = rng.random((n_mc, P_line.shape[1])) < P_line[s][None, :]
        out[s] = np.mean([max(lost(~f) - base, 0.0) for f in fails])
    return out


def annual_from_events(values, sim_year, n_years):
    return np.bincount(np.asarray(sim_year, int), weights=np.asarray(values, float), minlength=n_years)


def rp_table(annual, rps=(10, 25, 50, 100, 250, 500)):
    return {T: float(np.quantile(annual, 1 - 1 / T)) for T in rps if T <= len(annual)}


def exceedance_curve(annual):
    a = np.sort(np.asarray(annual))[::-1]
    rank = np.arange(1, len(a) + 1)
    return pd.DataFrame({'annual_value': a, 'return_period_yr': len(a) / rank})
def build_risk_context(dirs, storm_10min_corr=1.0, kv_scope=None):
    """Load everything the risk engine needs: STORM hazard (03), towers with exposure (00/01), lines and population."""
    z = np.load(need(dirs['interim'] / 'hazard_storm_cells.npz', '03'), allow_pickle=False)
    th = pd.read_parquet(need(dirs['processed'] / 'towers_hazard.parquet', '03'))[['tower_id', 'cell']]
    towers = load_towers(dirs).merge(th, on='tower_id', how='inner')
    if kv_scope:
        towers = towers[towers.voltage_kv.isin(kv_scope)]
    lines, subs = load_network(dirs)
    lines = lines.sort_values('line_id').reset_index(drop=True)
    towers['line_pos'] = towers.line_id.map({lid: i for i, lid in enumerate(lines.line_id)})
    towers = towers.dropna(subset=['line_pos']).reset_index(drop=True)
    towers['line_pos'] = towers.line_pos.astype(int)
    n_sub = int(max(subs.sub_id.max(), lines.from_sub.max(), lines.to_sub.max()))
    sub_pop = np.zeros(n_sub)
    sub_pop[subs.sub_id.values.astype(int) - 1] = subs['pop'].values
    return dict(H=z['H'], years=z['sim_year'].astype(int), ids=z['storm_id'], n_years=int(z['n_years']),
                towers=towers, lines=lines, subs=subs, cell=towers.cell.values.astype(int),
                gfac=gust_factor(towers.exposure, 600) * storm_10min_corr, M=line_matrix(towers.line_pos.values, len(lines)),
                frm=lines.from_sub.values.astype(int) - 1, to=lines.to_sub.values.astype(int) - 1, n_sub=n_sub, sub_pop=sub_pop)


def risk_run(ctx, theta_t, beta, scale=1.0, n_mc=30, cost_cr=0.73, outage_days=7.0, kwh_pp_day=3.0, voll_inr=100.0, seed=7):
    """One full risk run over all STORM storms. Costs in INR crore. Returns events, annual totals and EAD."""
    theta_t = np.broadcast_to(np.asarray(theta_t, float), (len(ctx['cell']),))
    E, P_line, tsum = storm_probabilities(ctx['H'], ctx['cell'], ctx['gfac'], theta_t, beta, ctx['M'], scale)
    pop = outage_mc(P_line, ctx['frm'], ctx['to'], ctx['n_sub'], ctx['sub_pop'], n_mc=n_mc, seed=seed)
    ev = pd.DataFrame({'storm_id': ctx['ids'], 'sim_year': ctx['years'], 'exp_damaged_towers': E, 'exp_pop_disconnected': pop})
    ev['tower_cost_cr'] = E * cost_cr
    ev['ens_mwh'] = pop * outage_days * kwh_pp_day / 1000.0
    ev['outage_cost_cr'] = ev.ens_mwh * 1000.0 * voll_inr / 1e7
    ev['total_cost_cr'] = ev.tower_cost_cr + ev.outage_cost_cr
    cols = ['exp_damaged_towers', 'exp_pop_disconnected', 'tower_cost_cr', 'outage_cost_cr', 'total_cost_cr']
    annual = pd.DataFrame({c: annual_from_events(ev[c], ev.sim_year, ctx['n_years']) for c in cols})
    return dict(events=ev, annual=annual, P_line=P_line, tower_rate=tsum / ctx['n_years'],
                ead={c: float(annual[c].mean()) for c in cols})
# ==== end of shared functions ====

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

ROOT, DIRS = setup_project()

# ---------------- settings ----------------
AVG_S = 180                  # IMD best-track winds are 3-minute means
P_ENV = 1008.0               # environmental pressure (hPa) for Holland B
RMW_DEFAULT_KM = 30.0        # used where IBTrACS has no radius of maximum wind
DT_H = 0.25                  # track interpolation step (hours)
SAMPLE_KM = 150              # towers within this distance of the track enter the fragility fit
SAMPLE_KV = (132, 220)       # the OPTCL damage records cover 132 and 220 kV lines
VB_IS875 = 50.0              # IS 875 basic wind speed for coastal Odisha (3-s gust, 50-yr)
VD_OPTCL_10MIN = 50.0 / 1.375 * 1.00 * 1.08   # IS 802:1995 zone 5, reliability level 1, terrain cat 1 = 39.3 m/s
RAJ = {'collapse': 292.5 / 3.6, 'collapse_beta': 0.104, 'functional': 280.4 / 3.6, 'functional_beta': 0.088}

towers = load_towers(DIRS)
print(f'Towers: {len(towers):,}; exposure source: {towers.exposure_src.dropna().iloc[0]}')

## 1. Fani track

In [ ]:
f_ib = DIRS['processed'] / 'ibtracs_FANI_2019.csv'
if f_ib.exists():
    ib = pd.read_csv(f_ib, parse_dates=['ISO_TIME'])
    ib = ib.dropna(subset=['LAT', 'LON']).sort_values('ISO_TIME')
    wind_kt = ib.NEWDELHI_WIND.fillna(ib.get('WMO_WIND'))
    ib = ib[wind_kt.notna()]
    wind_kt = wind_kt[wind_kt.notna()]
    track = pd.DataFrame({'time': ib.ISO_TIME.values, 'lat': ib.LAT.values, 'lon': ib.LON.values,
                          'vmax': wind_kt.values * KT2MS,
                          'pc': ib.NEWDELHI_PRES.fillna(ib.get('WMO_PRES')).values,
                          'rmw': ib.USA_RMW.values * NMI2KM})
    TRACK_SRC = 'IBTrACS v04r01 (IMD/RSMC New Delhi winds, JTWC RMW)'
else:
    imd = pd.read_excel(DIRS['external'] / 'raj_network' / 'Step1 - WS_estimation_WDEmodel_fragility' / 'FANI.xlsx')
    track = pd.DataFrame({'time': pd.Timestamp('2019-04-26') + pd.to_timedelta(imd.iloc[:, 1], unit='h'),
                          'lat': imd.Lat, 'lon': imd.Long, 'vmax': imd['Vmax (kts)'] * KT2MS, 'pc': np.nan, 'rmw': np.nan})
    TRACK_SRC = 'IMD 3-hourly track from the Raj et al. repo (no pressure or RMW)'
track['t_h'] = (track.time - track.time.iloc[0]).dt.total_seconds() / 3600
n_rmw = track.rmw.notna().sum()
track['rmw'] = track.rmw.interpolate(limit_direction='both').fillna(RMW_DEFAULT_KM)
print(f'Track source: {TRACK_SRC}; {len(track)} fixes; RMW reported at {n_rmw} fixes (rest interpolated/default)')
print(f'Peak: {track.vmax.max() / KT2MS:.0f} kt ({track.vmax.max():.1f} m/s, 3-min); '
      f'min pressure: {np.nanmin(track.pc) if track.pc.notna().any() else "n/a"} hPa')

def run_fani(rmw_scale=1.0, b_fixed=None, plat=None, plon=None):
    tr = interp_track(track.t_h, track.lat, track.lon, track.vmax, track.rmw * rmw_scale, track.pc, dt_h=DT_H)
    return holland_max_wind(tr, plat, plon, p_env=P_ENV, b_fixed=b_fixed), tr

# distance from each tower to the track
trk = interp_track(track.t_h, track.lat, track.lon, track.vmax, track.rmw, track.pc, dt_h=DT_H)
def xyz(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    return np.c_[np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)]
d, _ = cKDTree(xyz(trk['lat'], trk['lon'])).query(xyz(towers.lat.values, towers.lon.values))
towers['dist_track_km'] = 2 * R_EARTH_KM * np.arcsin(np.clip(d / 2, 0, 1))
near = towers.dist_track_km <= 400
lf = np.argmin(np.hypot(trk['lat'] - 19.8, trk['lon'] - 85.8))
print(f'Track passes closest to Puri at t = {trk["t_h"][lf]:.1f} h; translation speed there '
      f'{np.hypot(trk["u"][lf], trk["v"][lf]) * 3.6:.0f} km/h; RMW used {trk["rmw"][lf]:.0f} km')

## 2. Wind and gust at every tower

In [ ]:
v180 = np.zeros(len(towers))
v180[near.values], _ = run_fani(plat=towers.lat.values[near.values], plon=towers.lon.values[near.values])
towers['v3min'] = v180
towers['gust3s'] = v180 * gust_factor(towers.exposure, AVG_S)
towers['v10min_equiv'] = towers.gust3s / gust_factor(towers.exposure, 600)
towers['gust_ratio_is875'] = towers.gust3s / VB_IS875
towers['v10_ratio_optcl'] = towers.v10min_equiv / VD_OPTCL_10MIN
cols = ['tower_id', 'line_id', 'voltage_kv', 'lat', 'lon', 'exposure', 'dist_track_km', 'v3min', 'gust3s',
        'v10min_equiv', 'gust_ratio_is875', 'v10_ratio_optcl', 'damaged', 'collapsed', 'damage_state', 'optcl_line_name']
towers[cols].to_parquet(DIRS['processed'] / 'fani_tower_wind.parquet', index=False)

dm = towers[towers.damaged]
print('Modelled 3-s gust at the 128 damaged towers (m/s):')
print(dm.gust3s.describe().round(1).to_string())
print(f'\nShare of damaged towers where the modelled gust exceeded the IS 875 basic wind speed (50 m/s): '
      f'{(dm.gust_ratio_is875 > 1).mean():.0%}')
print(f'Share where the 10-min equivalent exceeded OPTCL\'s IS 802:1995 design wind ({VD_OPTCL_10MIN:.1f} m/s): '
      f'{(dm.v10_ratio_optcl > 1).mean():.0%}')
print('\nBy line:')
print(dm.groupby('optcl_line_name').agg(towers=('tower_id', 'size'), collapsed=('collapsed', 'sum'),
      gust_median=('gust3s', 'median'), gust_max=('gust3s', 'max')).round(1).sort_values('towers', ascending=False).to_string())

## 3. Check against station observations
Station hourly means are roughly 10-minute averages, so the model's 3-minute wind is divided by the WMO factor G(180→600) before comparing. Gust reports are patchy; treat this as a sense check, not a calibration.

In [ ]:
obs_f = DIRS['processed'] / 'fani_station_obs_meteostat.csv'
if obs_f.exists():
    obs = pd.read_csv(obs_f, parse_dates=['time'])
    st = obs.groupby(['station', 'name', 'latitude', 'longitude']).agg(obs_max_wspd_kmh=('wspd', 'max'),
                                                                        obs_max_wpgt_kmh=('wpgt', 'max')).reset_index()
    vs, _ = run_fani(plat=st.latitude.values, plon=st.longitude.values)
    st['model_v10_kmh'] = vs / 1.09 * 3.6   # max 3-min → 10-min mean: WMO Table 1.1 in-land G(180 s, To = 600 s) = 1.09
    st['model_gust_kmh'] = vs * GUST3['in-land'][180] * 3.6
    st['dist_track_km'] = haversine_km(st.latitude.values, st.longitude.values, 19.8, 85.8)
    st = st.sort_values('dist_track_km')
    st.to_csv(DIRS['tables'] / '02_station_check.csv', index=False)
    print(st.head(12).round(1).to_string(index=False))
else:
    print('No station file from Notebook 00 (Meteostat step failed or skipped). Skipping.')

## 4. Fragility curves fitted on this wind field
Sample: 132/220 kV towers within 150 km of the track (damaged and undamaged). Outcome: any damage (128 towers) or collapse (87). Intensity measure: modelled 3-s gust at 10 m. Confidence bands come from 200 bootstrap resamples.

In [ ]:
smp = towers[(towers.dist_track_km <= SAMPLE_KM) & towers.voltage_kv.isin(SAMPLE_KV)]
print(f'Fit sample: {len(smp):,} towers, of which {smp.damaged.sum()} damaged and {smp.collapsed.sum()} collapsed '
      f'(of 128 / 87 in the records)')
frag = {}
for key, y in [('any_damage', smp.damaged.values), ('collapse', smp.collapsed.values)]:
    th, be, nll = fit_fragility(smp.gust3s.values, y)
    bt, bb = bootstrap_fragility(smp.gust3s.values, y, n=200)
    frag[key] = dict(theta=th, beta=be, theta_ci90=list(np.percentile(bt, [5, 95])), beta_ci90=list(np.percentile(bb, [5, 95])),
                     n_sample=int(len(smp)), n_events=int(y.sum()), predicted_events=float(frag_p(smp.gust3s, th, be).sum()),
                     im='3-s gust at 10 m (m/s), WMO exposure-based, from IMD 3-min track', source='Notebook 02 MLE on Fani')
    print(f'{key:10s}: median θ = {th:.1f} m/s ({th * 3.6:.0f} km/h), β = {be:.3f}; 90% CI θ {np.percentile(bt, 5):.1f}–{np.percentile(bt, 95):.1f}, '
          f'β {np.percentile(bb, 5):.2f}–{np.percentile(bb, 95):.2f}')
frag['raj_collapse'] = dict(theta=RAJ['collapse'], beta=RAJ['collapse_beta'], source='Raj, Kumar & Bhatia (IMD×1.58, model ensemble)')
frag['raj_functional'] = dict(theta=RAJ['functional'], beta=RAJ['functional_beta'], source='Raj, Kumar & Bhatia')
frag['settings'] = dict(track=TRACK_SRC, p_env=P_ENV, rmw_default_km=RMW_DEFAULT_KM, avg_s=AVG_S, sample_km=SAMPLE_KM,
                        sample_kv=list(SAMPLE_KV), exposure_src=str(towers.exposure_src.dropna().iloc[0]))
(DIRS['processed'] / 'fragility_params.json').write_text(json.dumps(frag, indent=2))
print(f'\nRaj et al. for comparison: collapse θ = {RAJ["collapse"]:.1f} m/s, β = 0.104 (different wind model; IM not identical)')

# Observed vs fitted damage fraction by gust bin
bins = np.arange(0, smp.gust3s.max() + 5, 5)
smp_b = smp.assign(bin=pd.cut(smp.gust3s, bins))
tab = smp_b.groupby('bin', observed=True).agg(towers=('tower_id', 'size'), damaged=('damaged', 'sum'), collapsed=('collapsed', 'sum'))
tab['frac_damaged'] = tab.damaged / tab.towers
tab['frac_collapsed'] = tab.collapsed / tab.towers
tab.to_csv(DIRS['tables'] / '02_damage_by_gust_bin.csv')
print(tab[tab.towers > 0].round(3).to_string())

## 5. Sensitivity of the fit
Because the fragility is fitted on our own wind field, errors in the wind model are partly absorbed into θ. What matters is that Notebooks 03–05 use the same model. This table shows how far θ moves with the main assumptions.

In [ ]:
rows = []
for lab, kw, expo_override in [('base', {}, None), ('RMW × 0.7', {'rmw_scale': 0.7}, None), ('RMW × 1.3', {'rmw_scale': 1.3}, None),
                               ('Holland B = 1.2', {'b_fixed': 1.2}, None), ('Holland B = 2.0', {'b_fixed': 2.0}, None),
                               ('all towers in-land (Raj-style 1.58)', {}, 'in-land')]:
    v, _ = run_fani(plat=smp.lat.values, plon=smp.lon.values, **kw)
    g = v * gust_factor([expo_override] * len(smp) if expo_override else smp.exposure, AVG_S)
    th, be, _ = fit_fragility(g, smp.damaged.values)
    rows.append(dict(case=lab, theta=round(th, 1), beta=round(be, 3), gust_at_damaged_median=round(np.median(g[smp.damaged.values]), 1)))
sens = pd.DataFrame(rows)
sens.to_csv(DIRS['tables'] / '02_fragility_sensitivity.csv', index=False)
print(sens.to_string(index=False))

## 6. Figures

In [ ]:
lon_g, lat_g = np.meshgrid(np.arange(85.0, 86.6, 0.01), np.arange(19.5, 20.8, 0.01))
vg, _ = run_fani(plat=lat_g.ravel(), plon=lon_g.ravel())
gg = (vg * GUST3['in-land'][AVG_S]).reshape(lat_g.shape)
try:
    gpkg = DIRS['processed'] / 'odisha_network.gpkg'
    lines = gpd.read_file(gpkg, layer='lines'); coast = gpd.read_file(gpkg, layer='coastline')
except Exception:
    lines = coast = None

fig, ax = plt.subplots(1, 2, figsize=(16, 7.5))
cf = ax[0].contourf(lon_g, lat_g, gg, levels=np.arange(30, 95, 5), cmap='YlOrRd', alpha=0.85)
plt.colorbar(cf, ax=ax[0], label='3-s gust, in-land exposure (m/s)')
ax[0].contour(lon_g, lat_g, gg, levels=[VB_IS875], colors='k', linewidths=1.5)
if lines is not None:
    lines.plot(ax=ax[0], color='grey', lw=0.6); coast.plot(ax=ax[0], color='navy', lw=1)
d = towers[towers.damaged]
ax[0].scatter(d[d.collapsed].lon, d[d.collapsed].lat, marker='x', c='k', s=18, label='collapsed')
ax[0].scatter(d[~d.collapsed].lon, d[~d.collapsed].lat, marker='^', facecolors='none', edgecolors='k', s=18, label='partial')
ax[0].plot(trk['lon'], trk['lat'], color='purple', lw=1.5, label='Fani track')
ax[0].set_xlim(85.0, 86.5); ax[0].set_ylim(19.5, 20.7); ax[0].set_aspect('equal'); ax[0].legend(fontsize=8)
ax[0].set_title('Fani: modelled peak 3-s gust (black contour = 50 m/s IS 875 basic wind speed)')

x = np.linspace(20, 110, 200)
for key, c in [('any_damage', 'tab:red'), ('collapse', 'k')]:
    ax[1].plot(x, frag_p(x, frag[key]['theta'], frag[key]['beta']), c=c, label=f"{key}: θ={frag[key]['theta']:.0f} m/s, β={frag[key]['beta']:.2f}")
ax[1].plot(x, frag_p(x, RAJ['collapse'], RAJ['collapse_beta']), '--', c='grey', label='Raj et al. collapse')
t = tab[tab.towers >= 5]
mid = [b.mid for b in t.index]
ax[1].scatter(mid, t.frac_damaged, c='tab:red', s=np.clip(t.towers, 10, 200), alpha=0.5, label='observed share damaged (size = towers)')
ax[1].scatter(mid, t.frac_collapsed, c='k', s=np.clip(t.towers, 10, 200), alpha=0.4)
ax[1].axvline(VB_IS875, c='b', ls=':', label='IS 875 Vb = 50 m/s')
ax[1].set_xlabel('modelled 3-s gust at 10 m (m/s)'); ax[1].set_ylabel('probability'); ax[1].legend(fontsize=8)
ax[1].set_title('Fragility fitted on Fani (132/220 kV)')
plt.tight_layout(); fig.savefig(DIRS['figures'] / '02_fani_wind_fragility.png', dpi=200); plt.show()